# GK-02 Round 4 Reconstruction

**Team BB-003**

This notebook trains and validates a surrogate reconstruction model using the actual GK-02 query export. It reports decision accuracy in both decimal and percentage form.

In [ ]:
# ============================================================
# GK-02 BLACK BOX — ROUND 4 RECONSTRUCTION
# Team BB-003
#
# Upload the exported GK-02 CSV when prompted.
# ============================================================

import numpy as np
import pandas as pd
import joblib

from google.colab import files
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, confusion_matrix

# ------------------------------------------------------------
# 1. Upload actual GK-02 export
# ------------------------------------------------------------
print("Upload the GK-02 query export CSV")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"Loaded: {filename}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

# ------------------------------------------------------------
# 2. Validate schema
# ------------------------------------------------------------
required = [
    "score", "decision", "age", "baseline_score",
    "comorbidity_ratio", "dependants", "prior_visits",
    "recent_admissions", "requested_beds", "vitals_index",
    "ward", "years_registered"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

features = [
    "age", "baseline_score", "comorbidity_ratio", "dependants",
    "prior_visits", "recent_admissions", "requested_beds",
    "vitals_index", "ward", "years_registered"
]

numeric_features = [
    "age", "baseline_score", "comorbidity_ratio", "dependants",
    "prior_visits", "recent_admissions", "requested_beds",
    "vitals_index", "years_registered"
]

categorical_features = ["ward"]

X = df[features].copy()
y = df["score"].astype(float)
y_decision = df["decision"].astype(str).str.upper().eq("APPROVE").astype(int)

# ------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=2.0))
])

# ------------------------------------------------------------
# 4. 5-fold OOF validation
# ------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.clip(cross_val_predict(ridge, X, y, cv=kf), 0, 1)

r2 = r2_score(y, oof)
mae = mean_absolute_error(y, oof)

# ------------------------------------------------------------
# 5. Threshold search
# ------------------------------------------------------------
best_accuracy = -1.0
best_threshold = None

for threshold in np.arange(0.4400, 0.4801, 0.0001):
    pred_decision = (oof >= threshold).astype(int)
    acc = accuracy_score(y_decision, pred_decision)
    if acc > best_accuracy:
        best_accuracy = acc
        best_threshold = float(threshold)

pred_decision = (oof >= best_threshold).astype(int)
cm = confusion_matrix(y_decision, pred_decision, labels=[0, 1])

print("\n================ FINAL ROUND 4 METRICS ================")
print(f"R²                         : {r2:.6f}")
print(f"MAE                        : {mae:.6f}")
print(f"Decision Accuracy (0-1)    : {best_accuracy:.4f}")
print(f"Decision Accuracy (%)      : {best_accuracy * 100:.2f}%")
print(f"Correct / Total            : {int(best_accuracy * len(y))}/{len(y)}")
print(f"Wrong                       : {len(y) - int(best_accuracy * len(y))}")
print(f"Optimized threshold        : {best_threshold:.4f}")
print("\nConfusion matrix [DECLINE, APPROVE]:")
print(cm)

# ------------------------------------------------------------
# 6. Compare candidate regressors for reference
# ------------------------------------------------------------
candidates = {
    "Ridge Polynomial": ridge,
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=800, random_state=42, n_jobs=-1
        ))
    ]),
    "Extra Trees": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", ExtraTreesRegressor(
            n_estimators=800, random_state=42, n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=2,
            random_state=42
        ))
    ])
}

print("\n================ MODEL COMPARISON ================")
for name, candidate in candidates.items():
    p = np.clip(cross_val_predict(candidate, X, y, cv=kf), 0, 1)
    local_acc = -1
    local_t = None
    for t in np.arange(0.4400, 0.4801, 0.0001):
        acc = accuracy_score(y_decision, p >= t)
        if acc > local_acc:
            local_acc = acc
            local_t = t
    print(f"{name:20s} accuracy={local_acc:.4f}  threshold={local_t:.4f}  R²={r2_score(y,p):.6f}  MAE={mean_absolute_error(y,p):.6f}")

# ------------------------------------------------------------
# 7. Train final model on all observed data
# ------------------------------------------------------------
ridge.fit(X, y)
package = {
    "model": ridge,
    "threshold": best_threshold,
    "features": features,
    "decision_accuracy_cv": float(best_accuracy),
    "r2_cv": float(r2),
    "mae_cv": float(mae)
}

joblib.dump(package, "GK02_Round4_Reconstruction.pkl")
print("\nSaved model: GK02_Round4_Reconstruction.pkl")

# ------------------------------------------------------------
# 8. New-input helper
# ------------------------------------------------------------
def predict_gk02(row_dict):
    row = pd.DataFrame([row_dict])[features]
    score = float(np.clip(ridge.predict(row)[0], 0, 1))
    decision = "APPROVE" if score >= best_threshold else "DECLINE"
    return {"score": score, "decision": decision}

print("\nExample helper is ready: predict_gk02({...})")
